In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [2]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 78.5 MB/s eta 0:00:00


In [3]:
from pathlib import Path
import sys

# Point to project root.
base = Path('/content/gdrive/MyDrive/ParkingSpaceDetection')

# Point to directories.
tif_dir = base / 'data' / 'source' / 'tif_tiles'
label_xml_dir = base / 'data' / 'source' / 'xml_labels'
png_dir = base / 'data' / 'png_tiles'
label_txt_dir = base / 'data' / 'txt_labels'
train_img_dir = base / 'data' / 'images' / 'train'
val_img_dir   = base / 'data' / 'images' / 'val'
train_lbl_dir = base / 'data' / 'labels' / 'train'
val_lbl_dir   = base / 'data' / 'labels' / 'val'


In [ ]:
sys.path.append(str(base))

from python.tif_to_png import tif_to_png

# Convert .tif to .png.
tif_to_png(tif_dir, png_dir)

In [ ]:
sys.path.append(str(base))

from python.xml_to_txt import xml_to_txt

# Convert .xml to .txt.
xml_to_txt(label_xml_dir, label_txt_dir)

In [ ]:
sys.path.append(str(base))

from python.train_val_data_split import create_dirs, split_data, move_pairs

# Create directories for training and validation data.
create_dirs(train_img_dir, val_img_dir, train_lbl_dir, val_lbl_dir)

# Split data into 80% train, 20% val.
train_imgs, val_imgs = split_data(png_dir)

# Move training and validation data into directories.
move_pairs(train_imgs, label_txt_dir, train_img_dir, train_lbl_dir)
move_pairs(val_imgs,   label_txt_dir, val_img_dir,   val_lbl_dir)

In [ ]:
sys.path.append(str(base))

from ultralytics import YOLO
from python.train_model import train_model

# Model.
model = YOLO("yolov9c.pt")
yaml_path = '/content/gdrive/MyDrive/ParkingSpaceDetection/data.yaml'

# Model training.
train_model(model, yaml_path)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.68 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/gdrive/MyDrive/ParkingSpaceDetection/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015,

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(96, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_runnin

In [ ]:
# Validate model
results = model.val(data=yaml_path)

Ultralytics 8.4.68 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 101 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.3±0.1 ms, read: 449.3±25.9 MB/s, size: 1426.1 KB)
val: Scanning /content/gdrive/MyDrive/ParkingSpaceDetection/data/labels/val.cache... 28 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 28/28 7.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.7s/it 3.4s
                   all         28        347      0.798      0.798      0.834      0.396
Speed: 18.4ms preprocess, 28.6ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to /content/runs/detect/val
